This code is for getting GNSS data from the UNAVCO servers.
Correct for the euler pole acording to different reference frames
Create files with the corrected data acording to the reference frame
Get linear rates from the data, and create a file with the rates
Plot corrected data

The code pre_process2 is exclusive for already referenced data obtained from OVSICORI

How to use:
Run all the def 
The last block of code is for parameters and constants
Define all the variables in the last block and run

Part 1
Import necessary libraries
Setting directory paths
Import Euler pole module

In [4]:
# ### 1. Importing necessary libraries
import io
import re
import os
import sys
import math
import requests
import numpy as np
import pandas as pd
from io import StringIO
from matplotlib import pyplot as plt
from scipy.signal import detrend


In [5]:
#### 2. Setting directory paths
basedir=os.getcwd()

mapdir=os.path.join(basedir,'processing/mapdata')
sys.path.append(os.path.join(basedir,'external_programs','euler_pole','euler_pole'))

## output directories
datadir=os.path.join(basedir,'timeseries_raw')
plotdir=os.path.join(basedir,'results/plots_timeseries')
ratedir=os.path.join(basedir,'results/rates')
from euler_pole import EulerPole as EP
from euler_pole import cart2sph

yearspernanosec=1/(86400e9*365.25) #convert numeric nanosecond into fractional year

## make necessary output directories if they don't exist
if not os.path.exists(datadir):
    os.mkdir(datadir)
if not os.path.exists(plotdir):
    os.mkdir(plotdir)
if not os.path.exists(ratedir):
    os.mkdir(ratedir)

In [6]:
def wcorr(df):
    """
    df contains dN,E,U and wN,E,U for dataset to determine
    the weighed covarience matrix
    """
    Q=np.array(df[['dN','dE','dU']])
    W=np.array(df[['wN','wE','wU']])
    QW=Q*W
    C=QW.T.dot(QW)/W.T.dot(W)  # covariance
    CorrNE=C[0][1]/(C[0][0]*C[1][1])**0.5
    CorrNU=C[0][2]/(C[0][0]*C[2][2])**0.5
    CorrEU=C[1][2]/(C[1][1]*C[2][2])**0.5
    return CorrNE,CorrNU,CorrEU

In [7]:
def getProcessedGNSS(Stats,analysisCenter,refFrame='igs14',processing='Uncleanded',refCoord='from_analysis_center',):
    '''
    Pull data from one of the automatic GNSS processing centers ('unr', or 'cwu') and save into relevant files.
    At present all flags above after analysisCenter only pertain to data pulled from 'cwu'.  All
    data pulled from 'unr' will assume defaults for their 'tenv' file format.
    
    Format 'tenv' is headerless.  Info on this is here: http://geodesy.unr.edu/gps_timeseries/README_tenv.txt

    Recently updated to use authentification tokens for UNAVCO: 9/27/2023
    Now using .pos files format version 1.1.1 (hopefully these remain more consistent.  Certainly much more data-rich)
    '''

    import requests
    import shutil
    from pathlib import Path
    from earthscope_sdk.auth.device_code_flow import DeviceCodeFlowSimple
    from earthscope_sdk.auth.auth_flow import NoTokensError
    import os
    # get, or update token if necessary
    #token_path='/Users/an77/tokens/'  # note that as of now, this will not work when you create your own file
    token_path= '/Users/lhylu/Library/Application Support/earthscope-cli/sso_tokens.json'  # note that as of now, this will not work when you create your own file
    print(token_path)


    device_flow = DeviceCodeFlowSimple(Path(token_path))
    try:
        # get access token from local path
        device_flow.get_access_token_refresh_if_necessary()
    except NoTokensError:
        # if no token was found locally, do the device code flow
        device_flow.do_flow()
    token = device_flow.access_token
    #file open(token_path,'r')
    #token=file.read()

    nStats=len(Stats)
    count=0
    # # if not os.path.exists(datadir):
    #     os.mkdir(datadir)
    
    for stat in Stats:
        count += 1
        print('Requesting data for: '+stat+'  ['+str(count)+'/'+str(nStats)+']') # usable output

        if analysisCenter == 'cwu':
            #URL='https://web-services.unavco.org/gps/data/position/' + stat +    '/v3?analysisCenter=' + analysisCenter +    '&referenceFrame=' + refFrame +    '&report=long&dataPostProcessing=' + processing +    '&refCoordOptio='+ refCoord
            #URL='https://data.unavco.org/archive/gnss/products/position/' + stat + '/' + stat + '.' + analysisCenter +  '.' + refFrame + '.csv'
            URL='https://data.unavco.org/archive/gnss/products/position/' + stat + '/' + stat + '.' + analysisCenter +  '.' + refFrame + '.pos'
            filename=os.path.join(datadir,stat + '.' + analysisCenter +  '.' + refFrame + '.pos')
            req = requests.get(URL, headers={"authorization": f"Bearer {token}"})

        elif analysisCenter == 'unr':
            URL='http://geodesy.unr.edu/gps_timeseries/tenv/IGS14/' + stat +    '.tenv'
            filename=os.path.join(datadir,stat+'.tenv')
            req = requests.get(URL) # get data
        else:
            print("ERROR:   analysic Center '"+ analysisCenter+"' not recognized.  Exiting.")
            exit(-1)
        if req.status_code == requests.codes.ok:
            # save the file
            # print("  File URL: "+ URL)
            url_content = req.content
            file = open(filename, 'wb')
            file.write(url_content)
            file.close()
        else:
            #problem occured
            print(f"failure: {req.status_code}, {req.reason}")
        


# #### Testing getCWUlocs
# analysisCenter = 'cwu'
# Stats=('CN20', 'EPZA', 'HUA2', 'IRZU', 'LEPA', 'LOLA', 'MOIN', 'PNEG', 'PTPA', 'PTPP', 'TGPM', 'VLCN') 
# result = getProcessedGNSS(Stats, analysisCenter)



In [ ]:
def GNSSTimeSeries2Pandas(stat, analysisCenter='cwu', datadir='timeseries_raw', refFrame='igs14'):
    """
    Convert data pulled from an automatic GNSS processing center ('unr', or 'cwu')
    into a standard pandas format.

    For 'cwu', the .pos file has a header; for 'unr', the .tenv format is headerless.
    """

    if not os.path.exists(datadir):
        print("Error:  Cannot find '"+datadir+"'.  Exiting.")
        exit(-1)

    if analysisCenter == 'cwu':
        filename = os.path.join(datadir, stat + '.' + analysisCenter + '.' + refFrame + '.pos')

        # find length of header
        with open(filename, 'r') as file:
            for num, line in enumerate(file, 1):
                if "End Field Description" in line:
                    hlength = num
                    break

        df = pd.read_csv(filename, delimiter=r"\s+", header=hlength)

        # Fix weird column name for date
        df = df.rename(columns={str('*YYYYMMDD'): 'YYYYMMDD'})

        # Parse date
        df['Date'] = pd.to_datetime(df['YYYYMMDD'], format='%Y%m%d')

        # Rename components and uncertainties
        Comps = ('N', 'E', 'U')
        for comp in Comps:
            df = df.rename(columns={f'd{comp}': f'{comp}pos'})
            df = df.rename(columns={f'S{comp.lower()}': f'{comp}err'})

        Cors = ('NE', 'EU', 'NU')
        for cor in Cors:
            df = df.rename(columns={f'R{cor.lower()}': f'{cor}cor'})  # lowercase in input file

        # Longitudes: 0–360 -> -180–180
        df['Lon'] = df['Elong'].where(df['Elong'] <= 180, df['Elong'] - 360)

    elif analysisCenter == 'unr':
        filename = os.path.join(datadir, stat + '.tenv')

        df = pd.read_csv(
            filename,
            header=None,
            delim_whitespace=True,
            names=[
                'STAT', 'Date', 'DecimalYear', 'MJD', 'GPSWeek', 'GPSWeekDay',
                'Epos', 'Npos', 'Upos',
                'AntHeight', 'Eerr', 'Nerr', 'Uerr',
                'NEcor', 'EUcor', 'NUcor'
            ]
        )

        # Parse date (UNR Date is YYYYMMDD)
        df['Date'] = pd.to_datetime(df['Date'], yearfirst=True)

        # For UNR, positions are already named Epos/Npos/Upos, so you can rename for consistency:
        df = df.rename(columns={
            'Npos': 'Npos',
            'Epos': 'Epos',
            'Upos': 'Upos'
        })
    else:
        print("ERROR:   analysisCenter '" + analysisCenter + "' not recognized.  Exiting.")
        exit(-1)

    # --- DEFINE NumDate ONCE: DECIMAL YEAR (continuous in time) ---
    year = df['Date'].dt.year.astype(float)
    doy  = df['Date'].dt.dayofyear.astype(float)
    df['NumDate'] = year + (doy - 1) / 365.25   # units: years

    #print(df)

    return df



#Testing GNSSTimeSeries2Pandas

# analysisCenter = 'cwu'
# Stats=('CN20', 'EPZA', 'HUA2', 'LEPA', 'PTPP', 'TGPM', 'VLCN') 
# for stat in Stats:
#     result = GNSSTimeSeries2Pandas(stat, analysisCenter=analysisCenter)
#     print(f"Station: {stat}")
#     print(result.columns.tolist())
#     print(result['Lon'].head(15))

# corrections made last edit:  oct 24 , 2025
# change long into decimal degrees from 0-360 to -180 to 180 to match the pole format 

In [9]:
def getCWUlocs(stat,analysisCenter='cwu',refFrame='igs14'): 
    '''
    directly pulls info from their station information page
    '''
    #creating pandas readable csv from the url
    file=os.path.join(datadir,stat+'.'+analysisCenter+'.'+refFrame+'.pos')
    lat=lon=height=-9999.9  # defaults
    with open(file,'r') as csv_file:
        for line in csv_file:
            if line.startswith("NEU Reference position"):
                words=line.split()
                lat,lon,height=float(words[4]),float(words[5]),float(words[6])
                # convert to degrees ### MODIFIED
                if lon > 180:
                    lon -= 360
                break
    return lat,lon,height

# ### Testing getCWUlocs
# analysisCenter = 'cwu'
# Stats=('CN20', 'EPZA', 'HUA2', 'LEPA', 'PTPP', 'TGPM', 'VLCN') 
# for stat in Stats:
#     result = getCWUlocs(stat)
#     print(f"Station: {stat}, Lat: {result[0]}, Lon: {result[1]}, Height: {result[2]}")


## last edited Oct 24, 2025
## change lon into decimal degrees from 0-360 to -180 to 180 to match the pole format and for plotting later

In [10]:
def ITRF2008(Plate):
    '''
    Reads the full 13-column ITRF2008 Table 3 (Absolute Plate Rotation Poles) and returns the Euler pole (lat, lon, rotation rate in deg/Ma).
    '''
    ITRF2008platesfile = os.path.join(basedir, 'processing', 'ITRF2008_table.txt')
    with open(ITRF2008platesfile, 'r', encoding='utf-8') as f:
        txt = f.read()
    txt = txt.replace('\u2212', '-')  # “−” → “-”
    buf = io.StringIO(txt)

    COLS = ['Plate', 'Abrev', 'NS','WX', 'WXERR', 'WY', 'WYERR', 'WZ', 'WZERR', 'W_DEGREE', 'W_DEGREE_ERR', 'WRMS_E', 'WRMS_N' ]
    ITRF2008_df = pd.read_csv(buf, skiprows=3, sep=r'\s+', header=None, engine='python', names=COLS)

    # clean up column values and formats
    ITRF2008_df['Abrev'] = ITRF2008_df['Abrev'].astype(str).str.strip().str.upper()
    num_cols = ['NS','WX','WXERR','WY','WYERR','WZ','WZERR','W_DEGREE','W_DEGREE_ERR','WRMS_E','WRMS_N']
    for c in num_cols:
        ITRF2008_df[c] = pd.to_numeric(ITRF2008_df[c], errors='coerce')

    # Select plate
    sel = ITRF2008_df[ITRF2008_df['Abrev'] == Plate.strip().upper()]
    if sel.empty:
        print(f"Plate '{Plate}' not found. Available: {ITRF2008_df['Abrev'].unique().tolist()}")
        return None

    row = sel.iloc[0]
    print(f"Working with plate {row['Abrev']} for ITRF2008 Euler poles")
    print(f'Selecting Wx: {row["WX"]}, Wy: {row["WY"]}, Wz: {row["WZ"]}')
    Plat, Plon, Prot = cart2sph(row['WX'], row['WY'], row['WZ'])
    Prot_degMa = Prot / 3.6  # convert mas/yr to deg/Ma
    Pole = EP(Plat, Plon, Prot_degMa) 

    print(f"Working with plate {row['Abrev']} for ITRF2008 Euler poles")
    #print("Values from table:")
    #print(ITRF2008_df[['WX', 'WY', 'WZ']].values)
    print(f"Lat: {Plat:.3f}°, Lon: {Plon:.3f}°, Rot: {Prot_degMa:.3f}°/Ma (result in table: {row['W_DEGREE']:.3f}°/Ma)")
    #print(ITRF2008_df)

    return Pole

##testing
result = ITRF2008('CA')
print(result)


Working with plate CA for ITRF2008 Euler poles
Selecting Wx: 0.049, Wy: -1.088, Wz: 0.664
Working with plate CA for ITRF2008 Euler poles
Lat: 31.370°, Lon: -87.421°, Rot: 0.354°/Ma (result in table: 0.354°/Ma)
EulerPole(lat=31.369664812738012, lon=-87.42132596768707, rot_velocity=0.3543208814578066)


In [11]:
def parseTimeSeries(df,analysisCenter='cwu'):
    Comps=('N','E', 'U')
    for comp in Comps:
        ycolComp=str(comp+'pos')
        df[ycolComp]=df[ycolComp]-df[ycolComp].mean()
    # establish rapid and final datasets 
    if analysisCenter == 'unr':
        dfFinal=df  # only final are used here
        dfRapid=pd.DataFrame()
    elif analysisCenter == 'cwu': 
        #dfF1=df.loc[df['Soln']=='final']          #final solutions
        #dfF2=df.loc[df['Soln']=='suppl']          #final solutions
        #dfF3=df.loc[df['Soln']=='suppf']          #final solutions
        dfFinal=df.loc[df['Soln']!='rapid'].reset_index() 
        #pd.concat([dfF1,dfF2,dfF3]).sort_index()
        dfRapid=df.loc[df['Soln']=='rapid'].reset_index()   #rapid solutions
    return dfFinal, dfRapid

Part 4
Create wieghts for covariance matrix
Calculate Linear Rates, create rate file, create corrected file per station
Plot corrected timeseries


In [12]:
def linearRates(Stats,ratesfile,platename,Euler,analysisCenter,yscale=1000):
    """
    Create velocity determinations from the data that is pulled from either the 'cwu' or 'unr' analysis centers.
    as well as a scale for plotting [default of yscale=1000 converts m to mm].
    In addition to writing the ratesfile, process will return a dictionary of fit values useful for futher plotting, etc.
    """
    ratesFile=open(os.path.join(ratedir,ratesfile), 'w')  # new velocity file
    ratesFile.write('# local Plate motion is defined as %s Plate \n' % (platename))
    ratesFile.write('# STAT        Lat      Long   Height      Nvel     Evel    Uvel     Nvel-loc Evel-loc  Nerr     Eerr     Uerr    NEcor   NUcor   EUcor         Sdate       Edate  InstallYear\n') 
    ratesFile.write('#           °        °      m          mm/yr    mm/yr    mm/yr    mm/yr    mm/yr     mm/yr    mm/yr    mm/yr                             YEAR-MO-DY  YEAR-MO-DY YEAR-MO-DY\n') 
    ratesFile.write('#--------------------------------------------------------------------------------------------------------------------------------------------------------------\n') 
    fitDict = {}  # dictionary of fit values for individual stations and components.  this info is returned for plotting later
    
    for stat in Stats:
        #creating pandas readable csv from the url
        if analysisCenter == 'cwu':
            lat, lon, height = getCWUlocs(stat)  
        else:
            print("Could not get coordinates from "+analysisCenter+" for "+stat+", setting to defaults.")
        df=GNSSTimeSeries2Pandas(stat,analysisCenter=analysisCenter) 
        print(f"\n=== {stat} ===")
        print(df['Soln'].value_counts())
        
        installYear = df['Date'].iloc[0].year

        dfFinal,dfRapid=parseTimeSeries(df,analysisCenter=analysisCenter)        
        print("FIRST DATE FINAL:", dfFinal['Date'].min())
        print("FIRST DATE RAPID:", dfRapid['Date'].min())

        dfFinal = dfFinal[dfFinal['Date'] >= pd.Timestamp("2013-01-01")].copy() # filter data from 2013 onwards
        dfFinal.reset_index(drop=True, inplace=True) #### ADDED!

        # save processed file per station  ### ALL of this was added
        ts_results_dir = os.path.join(basedir, 'results/processed_ts/earthscope')
        os.makedirs(ts_results_dir, exist_ok=True)
        out_file = os.path.join(ts_results_dir, f"{stat}_processed.csv")
        dfFinal.to_csv(out_file, index=False)
    
        xFND=dfFinal['NumDate']
        sp=0
        slopes=np.zeros(3) # store slopes and errors
        errs=np.zeros(3)
        dfFNEU=pd.DataFrame()
        Comps=('N','E', 'U')
        for comp in Comps:
            ycolComp=str(comp+'pos')
            wcolComp=str(comp+'err')
            sdate=dfFinal.Date[0].strftime("%Y-%m-%d")
            edate=dfFinal.Date[len(dfFinal)-1].strftime("%Y-%m-%d")
            yF=dfFinal[ycolComp]*yscale
            wghtF=1/dfFinal[wcolComp]/yscale
            # performing linear fits to data
            fit,var=np.polyfit(xFND,yF,1,w=wghtF, full=False, cov=True) # linear fit to data
            fitDict[stat+'-'+comp+'-fit'] = fit
            fitDict[stat+'-'+comp+'-var'] = var
            fity=np.polyval(fit,xFND)
            errs[sp]=np.sqrt(var[0][0])
            slopes[sp]=fit[0]
            dfFNEU['d'+comp]=(yF-fity) # detrended sln for covariance determination
            dfFNEU['w'+comp]=(wghtF)   #  weights 
            sp+=1
        CorrNE,CorrNU,CorrEU=wcorr(dfFNEU)
        ## 1) Obtener azimut y módulo de la velocidad de placa EN ITRF
        pAZ, pRate = Euler.velocity(lat, lon)   # pAZ en grados, pRate en mm/yr

        # 2) Pasar a componentes N y E de la placa EN ITRF
        plateN = pRate * np.cos(np.radians(pAZ))
        plateE = pRate * np.sin(np.radians(pAZ))

        # 3) Hacer placa-fijo: GNSS - placa
        vN_loc = slopes[0] - plateN   # slopes[0] en mm/yr
        vE_loc = slopes[1] - plateE
        print(plateN, plateE)



        print("Station %s (%.4f, %.4f):  N_loc = %6.2f mm/yr,  E_loc = %6.2f mm/yr"
        % (stat, lat, lon, vN_loc, vE_loc))
        ratesFile.write(
            f"{stat:4s} "
            f"{lat:8.4f} {lon:9.4f} {height:9.3f}  "
            f"{slopes[0]:9.3f} {slopes[1]:9.3f} {slopes[2]:9.3f}  "
            f"{errs[0]:9.3f} {errs[1]:9.3f} {errs[2]:9.3f}  "
            f"{CorrNE:7.4f} {CorrNU:7.4f} {CorrEU:7.4f}    "
            f"{sdate}  {edate}  {installYear}\n")

    ratesFile.close()
    return fitDict

In [13]:
def timeSeriesPlots(Stats, fitDict, analysisCenter, yscale=1000):
    """
    Create displacement timeseries from the data that is pulled from either the 'cwu' or 'unr' analysis centers.
    Requirs a Stats list of stations, the fit dictionary for those stations (created in linearRates() function), 
    as well as a scale for plotting [default of yscale=1000 converts m to mm].
    """
    # Plotting design options
    # errors
    csize=2; elw=0.8; ecol='k' # head size, width, color
    # markers 
    mec='k'; mew=0.8; mfmt='o'; msz='4'  #edge color, width, shape, size
    # final results
    falpha=1; fcol='blue'  # opacity, color
    # rapid results
    ralpha=0.5; rcol='red'  # opacity, color
    # grid
    galpha=0.5; gvwidth=0.5; ghwidth=1; gcol='gray'

    for stat in Stats:
        df=GNSSTimeSeries2Pandas(stat,analysisCenter=analysisCenter) 
        # split data into final and rapid, removing mean along the way
        dfFinal,dfRapid=parseTimeSeries(df,analysisCenter=analysisCenter)

        # Skip if empty
        if dfFinal.empty:
            print(f"[WARN] {stat}: no FINAL data available — skipping.")
            continue

        # Prepare plotting variables
        f, ax = plt.subplots(3, 1, figsize=(14, 8), sharey=False, sharex=True)
        f.tight_layout(h_pad=0)

        xF=dfFinal['Date']
        xFND=dfFinal['NumDate']
        if len(dfRapid) > 0 : 
            xR=dfRapid['Date']
        sp=0
        slopes=np.zeros(3) # store slopes and errors
        errs=np.zeros(3)
        dfFNEU=pd.DataFrame()
    
        Comps=('N','E', 'U')
        for comp in Comps:
            ycolComp=str(comp+'pos')
            wcolComp=str(comp+'err')
            yF=dfFinal[ycolComp]*yscale
            yeF=dfFinal[wcolComp]*yscale
            wghtF=1/dfFinal[wcolComp]/yscale
            if len(dfRapid) > 0 : 
                yR=dfRapid[ycolComp]*yscale
                yeR=dfRapid[wcolComp]*yscale
            
            # performing linear fits to data
            fit= fitDict[stat+'-'+comp+'-fit']
            var= fitDict[stat+'-'+comp+'-var']
            errs[sp]=np.sqrt(var[0][0])
            fity=np.polyval(fit,xFND)
            slopes[sp]=fit[0]
            dfFNEU['d'+comp]=(yF-fity) # detrended sln for covariance determination
            dfFNEU['w'+comp]=(wghtF)   #  weights 

            #plot
            ax[sp].plot(xF, fity)
            ax[sp].errorbar(x=xF, y=yF, yerr=yeF, 
                fmt=mfmt, ms=msz, capsize=csize, label='Final', mfc=fcol, mec=mec, mew=mew, 
                ecolor=ecol, elinewidth=elw, alpha=falpha)
            if len(dfRapid) > 0 : 
                ax[sp].errorbar(x=xR, y=yR, yerr=yeR, 
                    fmt=mfmt, ms=msz, capsize=csize, label='Rapid', mfc=rcol, mec=mec, mew=mew, 
                    ecolor=ecol, elinewidth=elw, alpha=ralpha)
            #plot labels and legend
            ax[sp].grid(axis='x', linestyle='-', color=gcol, linewidth=gvwidth, alpha=galpha)
            ax[sp].axhline(0, linestyle='-', color=gcol,linewidth=ghwidth, alpha=galpha) 
            if sp == 0 :
                ax[sp].legend(loc='upper left',fancybox=True, shadow=True)
                ax[sp].set_ylabel('North [mm]')
            if sp == 1 :
                ax[sp].set_ylabel('East [mm]')
            elif sp == 2 :
                ax[sp].set_ylabel('Vertical [mm]')
                ax[sp].set_xlabel('Date')
                [xmin,xmax,ymin,ymax]=plt.axis()
                ax[sp].text(xmin+(xmax-xmin)*.005, ymin+(ymax-ymin)*.01, 'Daily positions processed @ '+analysisCenter.upper(), 
                    horizontalalignment='left', 
                    verticalalignment='bottom')
                # put title atop as a last thing (includes values)
                ax[0].set_title('Costa Rica Sliver Project: '+stat+'  Rates: N=%.1f±%.1f, E=%.1f±%.1f, V=%.1f±%.1f [mm/yr]' % (slopes[0], errs[0],slopes[1], errs[1],slopes[2], errs[2]))
            sp+=1

        # Save plot
        f.savefig(
            os.path.join(plotdir, f"raw_ts/{stat}_TS_raw.png"),
            dpi=150, facecolor='white', bbox_inches='tight', pad_inches=0.5
        )
        plt.close('all')


Part 5
Define all the variables here and run

In [14]:
# 1. defaults
# select from 'cwu', 'unr', 'ovsi'
analysisCenter = 'cwu'

# Define stations for rates processing
#Stats=('CN20', 'EPZA', 'HUA2', 'IRZU', 'LAFE','LEPA', 'LOLA', 'MOIN', 'PNEG', 'PTPA', 'PTPP', 'TGPM', 'VLCN') # for rates
Stats=('CN20', 'EPZA', 'HUA2', 'LEPA', 'PTPP', 'TGPM','LAFE','PUJE', 'BON2') 

# # Define stations for common-mode processing
# meanList=('ABEJ','AROL', 'BIJA', 'BON2', 'BRBR', 'CABA', 'CAPO',) # for common mode

ratesfile='CR_rates_earthscope.txt' # this file will be created
#commonratesfile='CR_rates_common.txt' # this file will be created

mmpm=1000 # to get to mm displacements
UKFerr=[3,0.001,5] #[std measurement error, std process error, initial uncertainty in mm] 

### 2. get reference plate
# change def accordingly to the reference plate needed
#iPlate='CA'
#REF_Plate = ITRF2008(iPlate)
## local plate of reference
lPlate='CA'  
LOCAL_Plate = ITRF2008(lPlate)
# get local relative to it's refernce plate
#Local2REF= REF_Plate + LOCAL_Plate 
#Local2REF= LOCAL_Plate
#print(LOCAL_Plate, Local2REF)

# ### 3. pull data from data center
print("starting getProcessedGNSS")
getProcessedGNSS(Stats,analysisCenter=analysisCenter)

# ### 4. perform per-station processing and plot it
print("starting linear rates")
fitDict = linearRates(Stats, ratesfile, lPlate, LOCAL_Plate, analysisCenter, mmpm)
print("starting time series plots RAW")

timeSeriesPlots(Stats,fitDict,analysisCenter,mmpm)
# timeSeries2013(Stats, fitDict, analysisCenter, yscale=1000,)

# # ### 5. new common-mode processing
#dfCommon,fitCommon=commonMode(Stats,commonratesfile,analysisCenter,meanList)
# timeSeriesPlotsCommon(dfCommon,fitCommon,analysisCenter,meanList,yscale=mmpm)

# ### 6. Kalman Filtered PLots
# UKF(Stats,fitDict,analysisCenter,mmpm,UKFerr)




Working with plate CA for ITRF2008 Euler poles
Selecting Wx: 0.049, Wy: -1.088, Wz: 0.664
Working with plate CA for ITRF2008 Euler poles
Lat: 31.370°, Lon: -87.421°, Rot: 0.354°/Ma (result in table: 0.354°/Ma)
starting getProcessedGNSS
/Users/lhylu/Library/Application Support/earthscope-cli/sso_tokens.json
Requesting data for: CN20  [1/9]
Requesting data for: EPZA  [2/9]
Requesting data for: HUA2  [3/9]
Requesting data for: LEPA  [4/9]
Requesting data for: PTPP  [5/9]
Requesting data for: TGPM  [6/9]
Requesting data for: LAFE  [7/9]
Requesting data for: PUJE  [8/9]
Requesting data for: BON2  [9/9]
starting linear rates
      YYYYMMDD  HHMMSS  JJJJJ.JJJJ             X             Y             Z  \
0     20130311  120000     56362.5  848061.22148 -6.236541e+06  1.029563e+06   
1     20130312  120000     56363.5  848061.23501 -6.236541e+06  1.029563e+06   
2     20130313  120000     56364.5  848061.22980 -6.236541e+06  1.029563e+06   
3     20130314  120000     56365.5  848061.23442 -6.2